## **Aim**
To implement a program that detects suspicious domain names using basic domain reputation indicators.

## **Algorithm**
**Step 1:** Import `re`, `collections`, `datetime`, and `statistics` libraries.

**Step 2:** Create a list of domains to analyze with associated metadata (registration date, registrar, DNS records, etc.).

**Step 3:** Define suspicious indicators:
   - Recently registered domains (< 30 days)
   - Domains with high entropy (random-looking)
   - Domains with many subdomains
   - Typosquatting (similar to popular brands)
   - Use of suspicious TLDs (.tk, .ml, .ga, .cf, .gq, .xyz, etc.)
   - Domains with homograph characters
   - Domains with excessive hyphens or numbers
   - DGA-like patterns (consonant-heavy, no dictionary words)

**Step 4:** Score each domain based on indicators.

**Step 5:** Classify domains as suspicious, likely malicious, or benign.

**Step 6:** Generate a report with domain reputation scores.

In [1]:
import re
import json
import math
from datetime import datetime, timedelta
from collections import Counter

SUSPICIOUS_TLDS = {'.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.club', '.work', '.date', '.bid', '.loan', '.racing', '.download', '.stream', '.science', '.party', '.review', '.cricket', '.win', '.accountant', '.faith', '.trade'}

KNOWN_BRANDS = [
    'paypal', 'microsoft', 'apple', 'google', 'amazon', 'facebook',
    'instagram', 'twitter', 'netflix', 'dropbox', 'onedrive', 'icloud',
    'github', 'gitlab', 'bitbucket', 'slack', 'discord', 'telegram',
    'whatsapp', 'linkedin', 'yahoo', 'outlook', 'office365', 'adobe',
    'bank', 'chase', 'wellsfargo', 'bankofamerica', 'citibank',
    'paypal', 'stripe', 'square', 'venmo', 'coinbase', 'binance'
]

def calculate_entropy(text):
    """Calculate Shannon entropy of text"""
    if not text:
        return 0
    freq = Counter(text.lower())
    entropy = 0
    for count in freq.values():
        p = count / len(text)
        entropy -= p * math.log2(p)
    return entropy

def analyze_domain(domain_data):
    domain = domain_data["domain"].lower()
    reg_date = datetime.fromisoformat(domain_data["registration_date"])
    
    score = 0
    indicators = []
    
    # Extract main domain (remove subdomains)
    parts = domain.split('.')
    if len(parts) >= 2:
        main_domain = '.'.join(parts[-2:])
        subdomain = '.'.join(parts[:-2]) if len(parts) > 2 else ''
    else:
        main_domain = domain
        subdomain = ''
    
    # Suspicious TLD
    for tld in SUSPICIOUS_TLDS:
        if domain.endswith(tld):
            score += 25
            indicators.append(f"Suspicious TLD: {tld}")
            break
    
    # Recently registered (< 30 days)
    days_old = (datetime.now() - reg_date).days
    if days_old < 30:
        score += 20
        indicators.append(f"Recently registered ({days_old} days ago)")
    elif days_old < 90:
        score += 10
        indicators.append(f"Recently registered ({days_old} days ago)")
    
    # Brand impersonation in domain
    for brand in KNOWN_BRANDS:
        if brand in domain and not domain.startswith(brand + '.') and not domain == brand + '.com':
            score += 20
            indicators.append(f"Brand impersonation: {brand}")
            break
    
    # High entropy (DGA-like)
    entropy = calculate_entropy(domain.replace('.', '').replace('-', ''))
    if entropy > 3.5:
        score += 15
        indicators.append(f"High entropy (DGA-like): {entropy:.2f}")
    elif entropy > 3.0:
        score += 10
        indicators.append(f"Elevated entropy: {entropy:.2f}")
    
    # Excessive hyphens
    hyphen_count = domain.count('-')
    if hyphen_count >= 3:
        score += 10
        indicators.append(f"Excessive hyphens ({hyphen_count})")
    elif hyphen_count >= 2:
        score += 5
        indicators.append(f"Multiple hyphens ({hyphen_count})")
    
    # Excessive numbers
    digit_count = sum(c.isdigit() for c in domain)
    if digit_count >= 5:
        score += 10
        indicators.append(f"Excessive numbers ({digit_count})")
    
    # Many subdomains
    if len(parts) >= 4:
        score += 10
        indicators.append(f"Many subdomains ({len(parts)-2})")
    
    # Homograph characters (basic check for non-ASCII)
    if any(ord(c) > 127 for c in domain):
        score += 20
        indicators.append("Homograph/Unicode characters detected")
    
    # Determine risk
    if score >= 60:
        risk = "CRITICAL"
    elif score >= 40:
        risk = "HIGH"
    elif score >= 20:
        risk = "MEDIUM"
    elif score > 0:
        risk = "LOW"
    else:
        risk = "SAFE"
    
    return {
        "domain": domain,
        "score": score,
        "risk": risk,
        "indicators": indicators,
        "entropy": round(entropy, 2),
        "days_old": days_old,
        "registration_date": reg_date.strftime("%Y-%m-%d")
    }

def main():
    domains = [
        {"domain": "paypal-security-update.tk", "registration_date": "2026-08-15"},
        {"domain": "microsoft-login-verify.ml", "registration_date": "2026-08-10"},
        {"domain": "amazon-account-services.ga", "registration_date": "2026-08-05"},
        {"domain": "google-security-alert.xyz", "registration_date": "2026-07-20"},
        {"domain": "apple-id-verification.cf", "registration_date": "2026-07-25"},
        {"domain": "secure-banking-login.com", "registration_date": "2026-06-15"},
        {"domain": "netflix-account-update.net", "registration_date": "2026-05-01"},
        {"domain": "my-personal-blog.org", "registration_date": "2020-01-15"},
        {"domain": "company-website.com", "registration_date": "2015-03-10"},
        {"domain": "github.com", "registration_date": "2007-10-09"},
        {"domain": "xn--pple-43d.com", "registration_date": "2026-07-01"},  # Homograph
        {"domain": "secure-login-verify-update.net", "registration_date": "2026-07-15"},
        {"domain": "banking-services-online.xyz", "registration_date": "2026-08-01"},
        {"domain": "office365-security-alert.top", "registration_date": "2026-08-10"},
        {"domain": "random-string-xyz123.ml", "registration_date": "2026-08-12"},
    ]
    
    print("Analyzing domain reputation...")
    results = [analyze_domain(d) for d in domains]
    results.sort(key=lambda x: -x["score"])
    
    print(f"\n{'='*60}")
    print(f"SUSPICIOUS DOMAIN DETECTED REPORT")
    print(f"{'='*60}")
    print(f"Total domains analyzed: {len(results)}")
    
    print(f"\n--- DOMAIN REPUTATION SCORES ---")
    
    for i, r in enumerate(results, 1):
        print(f"\n{i}. [{r['risk']}] {r['domain']} (Score: {r['score']})")
        print(f"    Registered: {r['registration_date']} ({r['days_old']} days ago)")
        print(f"    Entropy: {r['entropy']}")
        print(f"    Indicators:")
        for ind in r["indicators"]:
            print(f"      - {ind}")
    
    # Summary
    from collections import Counter
    risk_counts = Counter(r["risk"] for r in results)
    print(f"\n--- SUMMARY ---")
    for risk in ["CRITICAL", "HIGH", "MEDIUM", "LOW", "SAFE"]:
        if risk in risk_counts:
            print(f"  {risk}: {risk_counts[risk]}")

if __name__ == "__main__":
    main()

Analyzing domain reputation...

SUSPICIOUS DOMAIN DETECTED REPORT
Total domains analyzed: 15

--- DOMAIN REPUTATION SCORES ---

1. [CRITICAL] paypal-security-update.tk (Score: 85)
    Registered: 2026-08-15 (5 days ago)
    Entropy: 3.54
    Indicators:
      - Suspicious TLD: .tk
      - Recently registered (5 days ago)
      - Brand impersonation: paypal
      - High entropy (DGA-like): 3.54
      - Multiple hyphens (2)

2. [CRITICAL] microsoft-login-verify.ml (Score: 85)
    Registered: 2026-08-10 (10 days ago)
    Entropy: 3.66
    Indicators:
      - Suspicious TLD: .ml
      - Recently registered (10 days ago)
      - Brand impersonation: microsoft
      - High entropy (DGA-like): 3.66
      - Multiple hyphens (2)

3. [CRITICAL] amazon-account-services.ga (Score: 85)
    Registered: 2026-08-05 (15 days ago)
    Entropy: 3.62
    Indicators:
      - Suspicious TLD: .ga
      - Recently registered (15 days ago)
      - Brand impersonation: amazon
      - High entropy (DGA-like): 3.

## **Result**
This the program successfully detects suspicious domain names using basic domain reputation indicators.